# Phase 15: MLflow Experiment Tracking

This project's experiments (Phases 4-9) were already run and their results saved as comparison CSVs and model files, so this notebook does two things:

1. **Retroactively logs every comparison you already ran** — class-imbalance techniques, supervised models, unsupervised models, the hybrid score, and (if you've run notebook 06) the final threshold analysis — into MLflow as proper tracked runs, so they show up in the MLflow UI with params and metrics instead of living only in CSVs.
2. **Registers the champion model** (LightGBM) in the MLflow Model Registry and tags it with a `champion` alias, so Phase 16 (monitoring) and any future retraining have one canonical place to load "the current production model" from — `models:/fraud-detection-champion@champion` — instead of a hardcoded file path.

Nothing here retrains anything. It's an honest record of experiments that already happened, which is exactly what MLflow is for when you adopt it partway through a project rather than from day one.

In [ ]:
import json
from pathlib import Path

import joblib
import mlflow
import mlflow.lightgbm
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().parent
MODELS = ROOT / "models"

# A local SQLite-backed store, not a plain file store: the Model Registry
# (used below to register and alias the champion model) needs a database
# backend, which the default `mlruns/` file store doesn't support.
mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlflow.db'}")
mlflow.set_experiment("fraud-detection-pipeline")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## Phase 15, Step 1: Log Phase 4 — Class Imbalance Comparison

One MLflow run per resampling technique, with its precision/recall/F1/ROC-AUC/PR-AUC as metrics and the technique name as a param. This is what lets you compare them later in the MLflow UI's parallel-coordinates view instead of re-reading a CSV.

In [ ]:
imbalance_df = pd.read_csv(MODELS / "class_imbalance_comparison.csv", index_col=0)
print(imbalance_df)

for technique, row in imbalance_df.iterrows():
    with mlflow.start_run(run_name=f"imbalance-{technique}"):
        mlflow.set_tags({"phase": "4-class-imbalance", "technique": technique})
        mlflow.log_param("resampling_technique", technique)
        for metric, value in row.items():
            mlflow.log_metric(metric.replace("-", "_"), float(value))

print(f"\nLogged {len(imbalance_df)} runs under phase=4-class-imbalance.")
print(f"Winner (by PR-AUC): {imbalance_df['PR-AUC'].idxmax()} -> carried into Phase 5.")

## Phase 15, Step 2: Log Phase 5 — Supervised Model Comparison

In [ ]:
supervised_df = pd.read_csv(MODELS / "supervised_model_comparison.csv", index_col=0)
print(supervised_df)

for name, row in supervised_df.iterrows():
    with mlflow.start_run(run_name=f"supervised-{name}"):
        mlflow.set_tags({"phase": "5-supervised-ml", "model_type": name})
        mlflow.log_param("model_type", name)
        mlflow.log_param("resampling_technique", imbalance_df["PR-AUC"].idxmax())
        for metric, value in row.items():
            mlflow.log_metric(metric.replace("-", "_"), float(value))

champion_name = supervised_df["PR-AUC"].idxmax()
print(f"\nLogged {len(supervised_df)} runs under phase=5-supervised-ml.")
print(f"Champion (by PR-AUC): {champion_name}")

## Phase 15, Step 3: Log Phase 6 — Unsupervised Comparison, and Phase 7 — Hybrid Score

The hybrid score's metrics are **recomputed here from the saved Isolation Forest + hybrid weights**, not copy-pasted from notebook 05's printed output, so this run's numbers are freshly verified rather than just re-stated.

In [ ]:
unsup_df = pd.read_csv(MODELS / "unsupervised_comparison.csv", index_col=0)
print(unsup_df)

for name, row in unsup_df.iterrows():
    with mlflow.start_run(run_name=f"unsupervised-{name}"):
        mlflow.set_tags({"phase": "6-unsupervised", "model_type": name})
        mlflow.log_param("model_type", name)
        for metric, value in row.items():
            mlflow.log_metric(metric.replace("-", "_"), float(value))

print(f"Logged {len(unsup_df)} runs under phase=6-unsupervised.")

In [ ]:
# Recompute the hybrid score fresh on the validation set to log genuinely-verified metrics.
X_val = pd.read_csv(ROOT / "data" / "processed" / "X_val.csv")
y_val = pd.read_csv(ROOT / "data" / "processed" / "y_val.csv").squeeze()
best_supervised_model = joblib.load(MODELS / "best_supervised_model.pkl")
iso_forest = joblib.load(MODELS / "isolation_forest.pkl")
hybrid_config = joblib.load(MODELS / "hybrid_config.pkl")

fraud_proba = best_supervised_model.predict_proba(X_val)[:, 1]
iso_raw = -iso_forest.score_samples(X_val)
iso_norm = np.clip(
    (iso_raw - hybrid_config["iso_score_min"]) / (hybrid_config["iso_score_max"] - hybrid_config["iso_score_min"]),
    0, 1
)
hybrid_risk = hybrid_config["supervised_weight"] * fraud_proba + hybrid_config["anomaly_weight"] * iso_norm

hybrid_pr_auc = average_precision_score(y_val, hybrid_risk)
hybrid_roc_auc = roc_auc_score(y_val, hybrid_risk)

with mlflow.start_run(run_name="hybrid-supervised-plus-anomaly"):
    mlflow.set_tags({"phase": "7-hybrid"})
    mlflow.log_params({
        "supervised_weight": hybrid_config["supervised_weight"],
        "anomaly_weight": hybrid_config["anomaly_weight"],
        "supervised_model": champion_name,
        "anomaly_model": "Isolation Forest",
    })
    mlflow.log_metric("PR_AUC", float(hybrid_pr_auc))
    mlflow.log_metric("ROC_AUC", float(hybrid_roc_auc))

print(f"Hybrid PR-AUC (freshly recomputed): {hybrid_pr_auc:.4f}")
print(f"LightGBM alone PR-AUC (from Phase 5): {supervised_df.loc[champion_name, 'PR-AUC']}")
print("-> matches notebook 05's reported 0.7545, confirming LightGBM alone remains the better choice.")

## Phase 15, Step 4: Log Phase 8/9 — Final Threshold Analysis (if available)

Only runs if you've already run `notebook 06`. If not, this is skipped with a clear message rather than failing — same graceful-fallback pattern as notebooks 07/08/the API.

In [ ]:
threshold_path = MODELS / "threshold_config.pkl"
eval_path = MODELS / "final_test_evaluation.csv"

if threshold_path.exists():
    threshold_config = joblib.load(threshold_path)
    with mlflow.start_run(run_name="final-threshold-selection"):
        mlflow.set_tags({"phase": "8-9-evaluation-threshold"})
        mlflow.log_param("final_threshold", threshold_config["final_threshold"])
        for cand_name, cand_val in threshold_config.get("candidates", {}).items():
            mlflow.log_param(f"candidate_{cand_name}", cand_val)
        if eval_path.exists():
            eval_df = pd.read_csv(eval_path, index_col=0)
            for metric, value in eval_df.iloc[0].items():
                try:
                    mlflow.log_metric(metric.replace("-", "_"), float(value))
                except (TypeError, ValueError):
                    pass
    print(f"Logged final threshold run: final_threshold={threshold_config['final_threshold']}")
else:
    print("models/threshold_config.pkl not found -> notebook 06 hasn't been run yet. "
          "Run it, then re-run this cell to log the final threshold decision.")

## Phase 15, Step 5: Register the Champion Model

Registers the actual `best_supervised_model.pkl` LightGBM model — logged with `mlflow.lightgbm.log_model` (not the generic sklearn flavor, since LightGBM's own flavor avoids a serialization issue with scikit-learn's newer default format) — under the name `fraud-detection-champion`, then tags the resulting version with a `champion` alias.

Using an **alias** rather than the older "Stage" system (Staging/Production) is the current MLflow convention — stages are deprecated as of MLflow 2.9+. `models:/fraud-detection-champion@champion` always resolves to whichever version currently holds that alias, so promoting a retrained model later is just moving the alias, not changing any code that loads it.

In [ ]:
from mlflow import MlflowClient

MODEL_NAME = "fraud-detection-champion"
X_test_sample = pd.read_csv(ROOT / "data" / "processed" / "X_test.csv").head(5)

with mlflow.start_run(run_name="register-champion"):
    mlflow.set_tags({"phase": "15-mlflow-registration", "model_type": champion_name})
    mlflow.log_params({
        "model_type": champion_name,
        "resampling_technique": imbalance_df["PR-AUC"].idxmax(),
    })
    for metric, value in supervised_df.loc[champion_name].items():
        mlflow.log_metric(metric.replace("-", "_"), float(value))
    model_info = mlflow.lightgbm.log_model(
        best_supervised_model,
        name="model",
        input_example=X_test_sample,
        registered_model_name=MODEL_NAME,
    )

client = MlflowClient()
latest_version = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'"))
client.set_registered_model_alias(MODEL_NAME, "champion", latest_version)

print(f"Registered '{MODEL_NAME}' version {latest_version}, aliased as 'champion'.")
print(f"Load it anywhere with: mlflow.lightgbm.load_model('models:/{MODEL_NAME}@champion')")

## Phase 15, Step 6: Verify the Registered Model Matches the Original Exactly

Loads the model back **from the registry** (not from the local `.pkl`) and checks its predictions match `best_supervised_model.pkl` exactly on real test transactions. This is the actual proof that registration didn't silently change anything — not just that the registration call succeeded.

In [ ]:
loaded_champion = mlflow.lightgbm.load_model(f"models:/{MODEL_NAME}@champion")

X_test = pd.read_csv(ROOT / "data" / "processed" / "X_test.csv")
original_proba = best_supervised_model.predict_proba(X_test)[:, 1]
registry_proba = loaded_champion.predict_proba(X_test)[:, 1]

assert np.allclose(original_proba, registry_proba, atol=1e-9), "Registry model doesn't match the original!"
print(f"Verified: the registered model reproduces best_supervised_model.pkl exactly on all {len(X_test):,} test transactions.")

## Phase 15, Step 7: View Results

Launch the MLflow UI to browse every run logged above, compare them side by side, and see the registered model:

```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Then open http://127.0.0.1:5000 — the experiment is called **fraud-detection-pipeline**, and the registered model is **fraud-detection-champion** (Models tab).

In [ ]:
summary = mlflow.search_runs(experiment_names=["fraud-detection-pipeline"])
print(f"Total runs logged: {len(summary)}")
print(summary["tags.mlflow.runName"].tolist())